# LLaMA-2 Final Model Training with LoRA
This notebook trains a LLaMA-2 7B model for sequence classification using a predefined set of optimal hyperparameters and LoRA for parameter-efficient fine-tuning. The dataset is split into training (80%), validation (10%), and test (10%) sets.

In [1]:
import pandas as pd
import numpy as np
import torch
import os
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    DataCollatorWithPadding
)
from datasets import Dataset
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report
)

/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
2025-07-10 11:25:10.400436: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-07-10 11:25:10.400532: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-07-10 11:25:10.401794: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:

In [2]:
import wandb
import huggingface_hub

os.environ["WANDB_PROJECT"] = "llama2_degendered_final"

wandb.login(key="1ad06854b00e224ae562a79fe3c4cc218a95c08c")
# huggingface_hub.login(token="YOUR_HF_TOKEN")

wandb.init()

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hice1/mwesley32/.netrc
wandb: Currently logged in as: mtwesley to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
# Model and Hyperparameter Configuration
model_name = "meta-llama/Llama-2-7b-hf"
model_cache_path = "../scratch/cache/llama2_degendered_final"
target_modules = ["q_proj", "v_proj", "k_proj", "o_proj"]

hyperparameters = {
    "learning_rate": 3.0e-05,
    "num_train_epochs": 5,
    "per_device_train_batch_size": 4,
    "weight_decay": 0.02,
    "lora_dropout": 0.15,
    "lora_r": 16,
    "lora_alpha": 48
}

In [4]:
# Data Preparation (80:10:10 Split)
# Using the 'combined_letters_degendered.csv' dataset as specified.
df = pd.read_csv("data/combined_letters_degendered.csv")[["full_text", "label"]].dropna()
df["label"] = df["label"].astype(int)

# First split: 80% train, 20% temp (for validation and test)
X_train, X_temp, y_train, y_temp = train_test_split(
    df["full_text"],
    df["label"],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

# Second split: 10% validation, 10% test from the temp set
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_cache_path)
tokenizer.pad_token = tokenizer.eos_token

In [5]:
# Tokenization Function
def tokenize(example):
    tokens = tokenizer(example["text"], truncation=True, padding=False, max_length=512)
    tokens["labels"] = example["label"]
    return tokens

# Create Hugging Face Datasets
train_dataset = Dataset.from_dict({"text": X_train.tolist(), "label": y_train.tolist()})
val_dataset = Dataset.from_dict({"text": X_val.tolist(), "label": y_val.tolist()})
test_dataset = Dataset.from_dict({"text": X_test.tolist(), "label": y_test.tolist()})

# Tokenize datasets
tokenized_train = train_dataset.map(tokenize, batched=True).remove_columns(["text"])
tokenized_val = val_dataset.map(tokenize, batched=True).remove_columns(["text"])
tokenized_test = test_dataset.map(tokenize, batched=True).remove_columns(["text"])

data_collator = DataCollatorWithPadding(tokenizer)

Map:   0%|          | 0/7189 [00:00<?, ? examples/s]

Map:   0%|          | 0/899 [00:00<?, ? examples/s]

Map:   0%|          | 0/899 [00:00<?, ? examples/s]

In [6]:
# Metrics Computation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    
    # Get classification report
    report = classification_report(labels, preds, output_dict=True, zero_division=0, target_names=['Female', 'Male'])
    
    # Flatten the report for easy logging
    metrics = {
        'accuracy': report['accuracy'],
        'macro_avg_precision': report['macro avg']['precision'],
        'macro_avg_recall': report['macro avg']['recall'],
        'macro_avg_f1': report['macro avg']['f1-score'],
        'weighted_avg_precision': report['weighted avg']['precision'],
        'weighted_avg_recall': report['weighted avg']['recall'],
        'weighted_avg_f1': report['weighted avg']['f1-score'],
        'female_precision': report['Female']['precision'],
        'female_recall': report['Female']['recall'],
        'female_f1': report['Female']['f1-score'],
        'female_support': report['Female']['support'],
        'male_precision': report['Male']['precision'],
        'male_recall': report['Male']['recall'],
        'male_f1': report['Male']['f1-score'],
        'male_support': report['Male']['support']
    }
    
    print("Confusion Matrix:\n", confusion_matrix(labels, preds))

    return metrics

In [7]:
# Model Initialization with LoRA
base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "Female", 1: "Male"},
    label2id={"Female": 0, "Male": 1},
    cache_dir=model_cache_path,
    device_map="auto"
)
base_model.config.pad_token_id = tokenizer.eos_token_id

lora_config = LoraConfig(
    r=hyperparameters["lora_r"],
    lora_alpha=hyperparameters["lora_alpha"],
    lora_dropout=hyperparameters["lora_dropout"],
    bias="none",
    task_type=TaskType.SEQ_CLS,
    target_modules=target_modules
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-2-7b-hf and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 16,785,408 || all params: 6,624,137,216 || trainable%: 0.2534


In [8]:
# Training Arguments
final_model_output_dir = "../scratch/final_llama2_degendered_model"
training_args = TrainingArguments(
    output_dir=final_model_output_dir,
    per_device_train_batch_size=hyperparameters["per_device_train_batch_size"],
    per_device_eval_batch_size=hyperparameters["per_device_train_batch_size"],
    num_train_epochs=hyperparameters["num_train_epochs"],
    learning_rate=hyperparameters["learning_rate"],
    weight_decay=hyperparameters["weight_decay"],
    fp16=True,
    save_strategy="epoch",
    logging_steps=50,
    report_to="wandb",
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="weighted_avg_f1",
    eval_strategy="epoch",
    save_total_limit=1,
    run_name="final_llama2_degendered_training"
)

In [9]:
# Trainer Initialization
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val, # Use validation set for in-training evaluation
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

/tmp/ipykernel_342085/4000249224.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
# Train the Model
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro Avg Precision,Macro Avg Recall,Macro Avg F1,Weighted Avg Precision,Weighted Avg Recall,Weighted Avg F1,Female Precision,Female Recall,Female F1,Female Support,Male Precision,Male Recall,Male F1,Male Support
1,0.732400,0.702918,0.667408,0.499925,0.499980,0.449003,0.572734,0.667408,0.581358,0.309091,0.061151,0.102102,278.000000,0.690758,0.938808,0.795904,621.000000
2,0.592200,0.721298,0.652948,0.523846,0.512361,0.491088,0.589840,0.652948,0.600591,0.350877,0.143885,0.204082,278.000000,0.696815,0.880837,0.778094,621.000000
3,0.448700,0.798632,0.682981,0.587323,0.540061,0.521734,0.634129,0.682981,0.627688,0.464646,0.165468,0.244032,278.000000,0.710000,0.914654,0.799437,621.000000
4,0.326800,0.989364,0.664071,0.555550,0.531340,0.517916,0.613082,0.664071,0.619191,0.404762,0.183453,0.252475,278.000000,0.706339,0.879227,0.783357,621.000000


Confusion Matrix:
 [[ 17 261]
 [ 38 583]]
Confusion Matrix:
 [[ 40 238]
 [ 74 547]]
Confusion Matrix:
 [[ 46 232]
 [ 53 568]]
Confusion Matrix:
 [[ 51 227]
 [ 75 546]]


In [ ]:
# Final Evaluation on the Test Set
print("--- Final Evaluation on Test Set ---")
test_results = trainer.evaluate(eval_dataset=tokenized_test, metric_key_prefix="test")

print("\nFinal Test Set Evaluation Results:")
print(test_results)

In [ ]:
# Save the Final Model and Tokenizer
trainer.save_model(final_model_output_dir)
tokenizer.save_pretrained(final_model_output_dir)
print(f"Final model and tokenizer saved to: {final_model_output_dir}")